# FineWeb-Edu + CorrectedCorrector Training

Trains a PDA circuit via CP-SAT, then a CorrectedCorrector on FineWeb-Edu.

**Steps:**
1. Download / cache FineWeb-Edu
2. Build BPE tokenizer
3. Train PDA circuit
4. Train CorrectedCorrector (gated delta, stacked SSD)

**Last run:** 2026-04-26

In [5]:
import sys
sys.path.insert(0, '/kaggle/working')
from circuit_lm.data import load_text, load_sequences



ModuleNotFoundError: No module named 'circuit_lm'

In [1]:
import sys
sys.path.insert(0, '.')
sys.path.insert(0, 'src')

from pathlib import Path
import time

from circuit_lm.data import load_text, load_sequences
from circuit_lm.io import save_model, load_model
from circuit_lm.tokenizer import Tokenizer
from circuit_lm.train_joint_pda_cpsat import train_joint_pda as train_pda
from src.hybrid import train_corrected_hybrid, CorrectedCorrector

ModuleNotFoundError: No module named 'circuit_lm'

In [ ]:
# ============================================================
# STEP 0: Download / Cache FineWeb-Edu
# ============================================================

DATASET_NAME = "HuggingFaceFW/fineweb-edu"
SUBSET      = "100K"      # try "10M" if you have disk space
CACHE_PATH  = "fineweb_cache.txt"

from datasets import load_dataset

if Path(CACHE_PATH).exists():
    print(f"Using cached data: {CACHE_PATH}")
    data_path = CACHE_PATH
else:
    print(f"Streaming {DATASET_NAME} ({SUBSET}) from HuggingFace...")
    t0 = time.time()
    
    if SUBSET.endswith("M"):
        num_rows = int(SUBSET[:-1]) * 1000 * 1000
    elif SUBSET.endswith("K"):
        num_rows = int(SUBSET[:-1]) * 1000
    else:
        num_rows = int(SUBSET)
    
    ds = load_dataset(DATASET_NAME, split="train", streaming=True)
    
    with open(CACHE_PATH, "w", encoding="utf-8") as f:
        for i, row in enumerate(ds):
            if i >= num_rows:
                break
            if i > 0 and i % 10000 == 0:
                print(f"  Written {i:,}/{num_rows:,} rows...")
            f.write(row["text"] + "\n")
    
    print(f"  Wrote {i+1:,} rows in {time.time()-t0:.1f}s -> {CACHE_PATH}")
    data_path = CACHE_PATH

In [ ]:
# ============================================================
# STEP 1: Build BPE Tokenizer
# ============================================================

VOCAB_SIZE  = 4096
BPE_MERGES = 2048

t0 = time.time()
text = load_text(data_path)
print(f"Text: {len(text):,} chars")

# Use first 5M chars for tokenizer to keep it fast
tok = Tokenizer.from_text(
    text[:5_000_000],
    vocab_size=VOCAB_SIZE,
    mode="bpe",
    bpe_merges=BPE_MERGES,
)
print(f"Tokenizer: vocab={tok.vocab_size}, mode={tok.mode}")
print(f"Took {time.time()-t0:.1f}s")

In [ ]:
# ============================================================
# STEP 2: Train PDA Circuit (CP-SAT)
# ============================================================

STATE_BITS = 12    # 4096 states
STACK_DEPTH = 4
STEPS = 60         # CP-SAT solver steps

sequences = load_sequences(data_path, tok)
print(f"Sequences: {len(sequences):,}, total tokens: {sum(len(s) for s in sequences):,}")

t0 = time.time()
stack_steps   = STEPS // 3
remaining     = STEPS - stack_steps
trans_steps  = remaining // 2
emit_steps   = remaining - trans_steps
print(f"Stack={stack_steps}, Trans={trans_steps}, Emit={emit_steps}")

circuit = train_pda(
    sequences=sequences,
    vocab_size=tok.vocab_size,
    state_bits=STATE_BITS,
    stack_depth=STACK_DEPTH,
    stack_steps=stack_steps,
    transition_steps=trans_steps,
    emission_steps=emit_steps,
)
print(f"Circuit done in {time.time()-t0:.1f}s")
print(f"Circuit: vocab={circuit.vocab_size}, states={circuit.num_states}")

In [ ]:
# ============================================================
# STEP 3: Save Circuit
# ============================================================

CIRCUIT_OUT = "circuit_fineweb.json"
save_model(circuit, tok, CIRCUIT_OUT)
print(f"Saved circuit to {CIRCUIT_OUT}")

In [ ]:
# ============================================================
# STEP 4: Train CorrectedCorrector
# ============================================================

EPOCHS       = 3
BATCH_SIZE   = 64
LR           = 1e-3
MAX_EXAMPLES = 50_000
EMBED_DIM    = 256
HIDDEN_DIM   = 512
NUM_SSD_LAYERS = 2

CORRECTOR_OUT = "corrector_fineweb.pt"

t0 = time.time()
hybrid = train_corrected_hybrid(
    circuit_path=CIRCUIT_OUT,
    data_path=data_path,
    output_path=CORRECTOR_OUT,
    num_epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    max_examples=MAX_EXAMPLES,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_ssd_layers=NUM_SSD_LAYERS,
)
print(f"Corrector training done in {time.time()-t0:.1f}s")
print(f"Saved to {CORRECTOR_OUT}")

In [ ]:
# ============================================================
# STEP 5: Quick Evaluation
# ============================================================

from circuit_lm.eval import evaluate

# Test on a small held-out chunk
test_text = text[:100_000]
test_ids  = tok.encode(test_text)

correct = 0
total   = 0
state   = 0
stack   = []

for i in range(1, min(len(test_ids)-1, 5000)):
    # Get circuit prediction
    hist = circuit.state_histogram(state)
    pred = max(range(len(hist)), key=lambda k: hist[k]) if hist else 0
    
    if pred == test_ids[i]:
        correct += 1
    total += 1
    
    # Step
    state = circuit.next_state(state, test_ids[i-1])

print(f"Circuit-only accuracy on test: {100*correct/total:.2f}%")

# Hybrid eval
correct_h = 0
state = 0
stack = []

for i in range(1, min(len(test_ids)-1, 5000)):
    ctx = test_ids[max(0,i-32):i]
    pred_h, info = hybrid.predict(ctx)
    if pred_h == test_ids[i]:
        correct_h += 1
    
    state = circuit.next_state(state, test_ids[i-1])

print(f"Hybrid accuracy on test: {100*correct_h/total:.2f}%")